# Route B recoverable K7 review candidate
Scientific amendment and GPU acceptance are PENDING. This draft cannot execute primary training or continuation.
Use only this matching candidate ZIP/receipt. ST-04N stays unchanged. Epoch checkpoints do not provide mid-epoch recovery.


In [ ]:
import json, hashlib, subprocess, sys, zipfile, os
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
cached = [name for name in sys.modules if name == 'research' or name.startswith('research.')]
assert not cached, 'Cached research imports: ' + repr(cached) + '. Restart the notebook kernel to clear imports; preserve extracted files and attempt records. Do not reload modules from another release.'
from pathlib import Path
from google.colab import drive

UPLOAD = Path('/content/drive/MyDrive/DeepAA_migration')
DRIVE_DATA = Path('/content/drive/MyDrive/HAM10000')  # Candidate, verified later.
DURABLE = Path('/content/drive/MyDrive/DeepAA_stability_v1_D020')  # Keep fixed across sessions.
PAYLOAD = Path('/content/deepaa_transfer_k7')
WORKSPACE = Path('/content/deepaa_workspace_k7')
LOCAL_DATA = Path('/content/HAM10000')
ENV = Path('/content/deepaa_env')
PRIMARY_LAUNCH_ENABLED = False
print('Primary launch disabled. Dataset location is not yet verified.')
assert not PAYLOAD.exists() and not WORKSPACE.exists(), 'Extraction/checkout exists. Preserve it; select new absent PAYLOAD and WORKSPACE paths for this corrected release. Never delete attempt records or change DURABLE to bypass a claim.'
drive.mount('/content/drive')


## Verify the transfer before executing packaged code
The senior receipt is the trust anchor. Compare its ZIP SHA-256 with the senior's review output. ZIP extraction rejects unsafe names and verifies every file. No download from the public repository is used.


In [ ]:
receipt = json.loads((UPLOAD/'TRANSFER_RECEIPT.json').read_text())
assert receipt['release'], 'STOP: disabled draft. Senior must commit/review and rebuild release package.'
archive = UPLOAD/receipt['archive']
with archive.open('rb') as f: actual_zip = hashlib.file_digest(f, 'sha256').hexdigest()
assert actual_zip == receipt['sha256'], 'Transfer SHA-256 mismatch'
assert not PAYLOAD.exists(), 'Extraction exists. Do not merge. Review or start a fresh runtime.'
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    assert len(names) == len(set(names))
    for name in names:
        assert z.getinfo(name).orig_filename == name, 'Unsafe normalized ZIP member'
        p = Path(name)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in name and ':' not in name
        assert ((z.getinfo(name).external_attr >> 16) & 0o170000) != 0o120000
    z.extractall(PAYLOAD)
package = json.loads((PAYLOAD/'PACKAGE.json').read_text())
def file_sha(p):
    with p.open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
assert file_sha(PAYLOAD/'PACKAGE.json') == receipt['package_manifest_sha256']
actual = {p.relative_to(PAYLOAD).as_posix(): {'bytes': p.stat().st_size, 'sha256': file_sha(p)} for p in PAYLOAD.rglob('*') if p.is_file() and p != PAYLOAD/'PACKAGE.json'}
assert actual == package['files']
assert package['release'] and package['head'] == receipt['head'] and not package['source_status']
print(json.dumps(receipt, indent=2))
sys.path.insert(0, str(PAYLOAD/'source_bytes'))
from research.code.colab_transfer import restore, run_visible
ROOT = restore(PAYLOAD, WORKSPACE)
print('Restored clean reviewed source with exact packaged bytes and verified D-019 receipt:', ROOT)


## Isolated pinned environment â€” stop on any incompatibility
This downloads Python 3.13.3 and the explicit candidate Linux packages. It does not use Colab's TensorFlow defaults. If a wheel/version or GPU driver is incompatible, **stop and paste the error**; do not downgrade or alter settings. GPU compatibility is not yet established.


In [ ]:
assert not ENV.exists(), 'Environment already exists; do not silently update it.'
run_visible([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'])
run_visible([sys.executable, '-m', 'uv', 'venv', '--python', '3.13.3', '--seed', str(ENV)])
PYTHON = str(ENV/'bin/python')
INSTALL_REPORT = WORKSPACE/'installation.json'
run_visible([PYTHON, '-m', 'pip', 'install', '--force-reinstall', '--report', str(INSTALL_REPORT), '-r', str(ROOT/'research/requirements_colab.txt')])
run_visible([PYTHON, '-m', 'pip', 'check'])
resolved = json.loads(INSTALL_REPORT.read_text())
lines = []
for item in resolved['install']:
    digest = item['download_info']['archive_info']['hashes']['sha256']
    lines.append(item['metadata']['name']+'=='+item['metadata']['version']+' --hash=sha256:'+digest)
(WORKSPACE/'requirements_resolved_linux.txt').write_text('\n'.join(sorted(lines))+'\n')
print('Target-specific exact wheel-hash lock:', WORKSPACE/'requirements_resolved_linux.txt')


## Verify Drive data and copy to runtime-local storage
Hashes all 10,015 JPEGs and metadata against the Windows-generated source inventory. Frozen split bytes determine membership/order. No dataset upload, split regeneration or Drive-original mutation occurs.


In [ ]:
from research.code.colab_runtime import prepare_images
spec = json.loads((PAYLOAD/'DATASET.json').read_text())
prepare_images(DRIVE_DATA, LOCAL_DATA, spec)
print('Drive and runtime-local image/metadata bytes verified:', len(spec['files']))


## Local orchestration fixtures on the target (no model training)
These tests check preservation, paths, partial-attempt refusal and existing workflow safety. GPU checks run only in the next cell.


In [ ]:
run_visible([PYTHON, '-B', '-m', 'pytest', 'research/tests/test_colab_migration.py', 'research/tests/test_colab_k7_single.py', 'research/tests/test_colab_k7_validation.py', 'research/tests/test_recoverable_k7.py', 'research/tests/test_workflow.py', 'research/tests/test_output_contract.py', 'research/tests/test_degenerate_recovery.py', '-q', '-p', 'no:cacheprovider'], cwd=ROOT, env=dict(os.environ, PYTHONDONTWRITEBYTECODE='1'))


## Route B GPU acceptance gate
Legacy K7 readiness does not validate the amended RNG or continuation workflow. Follow research/RECOVERY_GPU_ACCEPTANCE.txt. No automatic acceptance or primary launch.


In [ ]:
RECOVERY_GPU_ACCEPTANCE_ENABLED = False
raise RuntimeError('STOP: separate authorization and the Route B GPU acceptance procedure are required; legacy readiness is not sufficient.')


## Separately authorized future primary segment
Requires a clean approved release, accepted scientific amendment, reviewed target acceptance and exact segment authorization. Keep disabled during review.


In [ ]:
# PRIMARY TRAINING ? separate approved start or continuation only
PRIMARY_LAUNCH_ENABLED = False
FROZEN_K7_LATENT6_CONFIRMED = False
assert PRIMARY_LAUNCH_ENABLED, 'Primary launch/continuation is not authorized'
assert FROZEN_K7_LATENT6_CONFIRMED, 'Explicit latent6 acknowledgement required'
AUTHORIZATION = Path('/content/REVIEWED_ROUTE_B_SEGMENT.json')
REVIEWED_AUTHORIZATION_SHA256 = ''
assert len(REVIEWED_AUTHORIZATION_SHA256) == 64
run_visible([PYTHON,'-B','-u',str(ROOT/'research/tools/colab_recoverable_k7.py'),'--k', '7', '--latent-dimension', '6', '--seed', '113', '--payload',str(PAYLOAD),'--workspace',str(WORKSPACE),'--dataset',str(LOCAL_DATA),'--durable',str(DURABLE),'--authorization',str(AUTHORIZATION),'--reviewed-authorization-sha256',REVIEWED_AUTHORIZATION_SHA256], env=dict(os.environ,PYTHONHASHSEED='113',TF_DETERMINISTIC_OPS='1',PYTHONDONTWRITEBYTECODE='1',MPLBACKEND='Agg'),log=WORKSPACE/'route_b_segment.log')


## After success or interruption
This operation ends after ONE K7/latent6/seed113 attempt is validated, durably hash-verified and archived. It never launches a next seed or combines K5 results. Preserve all Drive records. If `QUEUE_ACTIVE.json`, `SINGLE_ACTIVE.json` or any K7/seed113 registration blocks execution, stop for senior review; do not delete records, take over, rerun or create a different project directory to bypass them. A runtime loss may lose the last unpersisted bytes. No automatic cleanup/retry or keep-alive is provided.

Return `single_run_result.json`, `REGISTRATION.json`, `DURABLE_COMPLETE.json`, validation and archive hashes. Later joint evaluation requires explicitly verified transfers and source/CPU/GPU provenance. No latest-folder selection or automatic aggregation occurs here.
